In [0]:
-- The Other Gold Tables are 1-To-1 Mapping with Silver Tables. Since, Silver Tables are already standardized and clean, and, have goof naming convention. No need to copy-paste the data to Gold Layer to show that these are Gold Tables.
-- So, the practical solution is to create Gold Views, instead of Gold Tables, so that the data is not replicated. The data from Silver Layer can be used.

In [0]:
CREATE OR REPLACE VIEW retail_oc.retail_gold.dim_customer 
AS
SELECT 
account_id AS customer_id,
account_name AS customer_name,
account_type AS customer_type,
billing_city,
billing_state,
billing_country,
phone_cleaned AS phone,
Website,
Industry,
annual_revenue,
number_of_employees,
description
FROM retail_oc.retail_silver.account
WHERE is_deleted = false
AND is_active = true;

In [0]:
CREATE OR REPLACE VIEW retail_oc.retail_gold.dim_product 
AS
SELECT 
product_id,
product_name,
category,
subcategory,
brand,
unit_price,
supplier_name,
launch_date,
updated_at
FROM retail_oc.retail_silver.product_catalog
WHERE is_active = true;

In [0]:
CREATE OR REPLACE TABLE retail_oc.retail_gold.dim_calendar
AS
WITH date_range AS (
  SELECT explode(sequence(
    date_sub(current_date(), 730),  -- 2 years ago (730 days)
    current_date(),
    interval 1 day
  )) AS date
)
SELECT 
  date,
  year(date) AS year,
  quarter(date) AS quarter,
  month(date) AS month,
  date_format(date, 'MMMM') AS month_name,
  weekofyear(date) AS week_of_year,
  day(date) AS day_of_month,
  dayofweek(date) AS day_of_week,
  date_format(date, 'EEEE') AS day_name,
  CASE WHEN dayofweek(date) IN (1, 7) THEN true ELSE false END AS is_weekend,
  CASE WHEN dayofweek(date) = 1 THEN 'Sunday'
       WHEN dayofweek(date) = 2 THEN 'Monday'
       WHEN dayofweek(date) = 3 THEN 'Tuesday'
       WHEN dayofweek(date) = 4 THEN 'Wednesday'
       WHEN dayofweek(date) = 5 THEN 'Thursday'
       WHEN dayofweek(date) = 6 THEN 'Friday'
       WHEN dayofweek(date) = 7 THEN 'Saturday'
  END AS day_of_week_name,
  concat(year(date), '-Q', quarter(date)) AS year_quarter,
  concat(year(date), '-', lpad(month(date), 2, '0')) AS year_month
FROM date_range
ORDER BY date;

In [0]:
-- I want to create a Metric View with name as retail_metric in schema "retail_semantic"
-- Read the follwoing table schema and sample data:
-- retail_oc.retail_gold.fact_sales
-- retail_oc.retail_gold.dim_customer
-- retail_oc.retail_gold.dim_product
-- retail_oc.retail_gold.dim_calendar
-- Decide the dimension and measure based on the data and then create as a metric view